# Phase 2 — Dataset (WikiText-103)

**Run only after Phase 1 `check_environment` passed (0 failed).**

Settings (right panel):
- Accelerator: GPU T4 x2 or P100 (CPU also OK for Phase 2 — no training yet)
- Persistence: **Files Only**
- Internet: **On**

This notebook:
1. Re-attaches / clones the repo
2. Runs unit tests (no download)
3. Downloads WikiText-103 and builds the **paper article split** 28,475 / 60 / 60
4. Tokenizes with GPT-2, packs sequences, flags sensitive tokens
5. Writes artefacts under `data/wikitext103/`

After it finishes: **Save Version → Save & Run All (Commit)** so data survives.

## Cell 0 — clone / update repo

In [ ]:
import os, sys

REPO = 'https://github.com/Yash-0525/sentiment_bias_project.git'
BRANCH = 'arena/01a0a078-sentiment-bias-project'
DEST = '/kaggle/working/sentiment_bias_project'

if os.path.isdir(DEST):
    print('pulling latest')
    !cd {DEST} && git fetch --quiet origin && git checkout {BRANCH} && git pull --quiet origin {BRANCH}
else:
    !git clone --branch {BRANCH} --single-branch --quiet {REPO} {DEST}

sys.path.insert(0, DEST)
%cd {DEST}
!git log -1 --oneline
from src import paths
paths.bootstrap(verbose=True)
print('ROOT', paths.ROOT)

## Cell 1 — unit tests (no download, must be instant PASS)

In [ ]:
!python tests/test_phase2_unit.py

**Expected:** `ALL PASS`

If this fails, stop and paste the traceback — do not run Cell 2.

## Cell 2 — FULL Phase 2 pipeline (download + preprocess)

Takes roughly **5–15 minutes** on Kaggle (mostly HF download + tokenize).

Uses:
- **ORIGINAL PAPER** article split 28,475 / 60 / 60
- **STUDENT** tokenizer `gpt2`, seq_len **256** (paper used 512)

For a 2-minute smoke test instead, add `--max-train-articles 500` to the command.

In [ ]:
# FULL RUN (recommended)
!python -m src.data_preprocessing \
    --out-dir /kaggle/working/sentiment_bias_project/data/wikitext103 \
    --seq-len 256 \
    --tokenizer gpt2

# SMOKE TEST alternative (uncomment to use instead of full run):
# !python -m src.data_preprocessing \
#     --out-dir /kaggle/working/sentiment_bias_project/data/wikitext103_smoke \
#     --seq-len 256 \
#     --tokenizer gpt2 \
#     --max-train-articles 500

## Cell 3 — verify artefacts + print sanity report

In [ ]:
from pathlib import Path
import json

root = Path('/kaggle/working/sentiment_bias_project/data/wikitext103')
assert root.is_dir(), root

required = [
    'phase2_summary.txt',
    'phase2_sanity_report.json',
    'phase2_config.json',
    'processed/articles_train.jsonl',
    'processed/articles_validation.jsonl',
    'processed/articles_test.jsonl',
    'processed/sequences_train.jsonl',
    'processed/sequences_validation.jsonl',
    'processed/sequences_test.jsonl',
    'processed/sensitive_indices_train.json',
    'processed/tokenizer_info.json',
]
print('=== required files ===')
missing = []
for rel in required:
    p = root / rel
    ok = p.is_file()
    size = f'{p.stat().st_size/1024**2:.2f} MB' if ok else 'MISSING'
    print(f'  {"OK" if ok else "XX"}  {rel:45s}  {size}')
    if not ok:
        missing.append(rel)

print()
print((root / 'phase2_summary.txt').read_text())

rep = json.loads((root / 'phase2_sanity_report.json').read_text())
print('=== checkpoint numbers ===')
print('article_counts:', rep['article_counts'])
print('sequence_counts:', rep['sequence_counts'])
print('sensitive:', json.dumps(rep['sensitive_token_stats'], indent=2))

ac = rep['article_counts']
# Hard gates for FULL run (not smoke)
if ac['train'] >= 20000:
    assert ac['train'] == 28475, ac
    assert ac['validation'] == 60, ac
    assert ac['test'] == 60, ac
    print('\nPAPER SPLIT MATCHES: 28475 / 60 / 60')
else:
    print('\nSMOKE / reduced train detected — OK if intentional')

assert not missing, f'missing files: {missing}'
print('\nPHASE 2 VERIFICATION: PASS')
print('Now: Save Version → Save & Run All (Commit) to keep data/')

## Cell 4 — end-of-session size check

In [ ]:
from src import paths
import os
root = paths.ROOT
print(paths.disk_report())
print(paths.project_size())
data = root / 'data' / 'wikitext103'
if data.is_dir():
    total = sum(f.stat().st_size for f in data.rglob('*') if f.is_file())
    print(f'data/wikitext103 = {total/1024**2:.1f} MB')